# 00 · Organização do Ambiente
### Atividade 2 · Projeto CineData Analytics (base combinada TMDB/IMDb)

Objetivos deste notebook:
1. Definir a estrutura de catálogo e schemas que vamos usar (Inputs → Bronze → Silver → Gold)
2. Criar o `catalog` e o **Volume** `inputs`, que recebe os arquivos CSV da pasta Inputs
3. Validar que os arquivos de origem estão disponíveis

Os schemas `silver` e `gold` são criados nos notebooks das respectivas camadas.

## 1. Parâmetros do ambiente

In [0]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "cinedata_analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

inputs_path = f"/Volumes/{catalog}/{bronze_schema_name}/inputs"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"inputs_path: {inputs_path}")

catalog: cinedata_analytics
bronze_schema: cinedata_analytics.bronze
silver_schema: cinedata_analytics.silver
gold_schema: cinedata_analytics.gold
inputs_path: /Volumes/cinedata_analytics/bronze/inputs


## 2. Criação do Catalog e do Volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.inputs")

DataFrame[]

## 3. Upload dos arquivos de origem

Os 5 arquivos foram enviados para a pasta **Inputs** em **Catalog** → `cinedata_analytics` → `bronze` → `inputs`
- `movies_info_TMDB_IMDB.csv`
- `movies_financials_IMDB_TMDB.csv`
- `movies_metrics_IMDB_TMDB.csv`
- `credits_and_tags_IMDB_TMDB.csv`
- `movies_reviews.csv`

A cotação do dólar não vem de arquivo: ela é extraída da API do Banco Central no notebook 01.

## 4. Validação do Volume

In [0]:
expected_files = [
    "movies_info_TMDB_IMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_reviews.csv",
]

try:
    existing = {f.name for f in dbutils.fs.ls(inputs_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{inputs_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    raise FileNotFoundError(f"Arquivos não encontrados no volume: {', '.join(missing)}")
else:
    print("[OK] Todos os arquivos esperados estão no volume:")
    for f in dbutils.fs.ls(inputs_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")

[OK] Todos os arquivos esperados estão no volume:
  - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - movies_info_TMDB_IMDB.csv (33129.0 KB)
  - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - movies_reviews.csv (1878.4 KB)
